In [0]:
%pip install flagembedding

In [0]:
# ============================================================================
# NOTEBOOK: unblinding_analysis   (NEW notebook — read-only consumer)
#
# Purpose: use the similarity-search stack built in build_reference_glossary
# to assemble the "inadvertent unblinding" cohort of deviations, then profile
# root causes, CAPAs, impacted programs, and stated safety/data-integrity impact.
#
# Cluster: CPU is fine EXCEPT Cell 2 (query embedding needs the same BGE-M3 as
#          indexing). If you want to avoid GPU entirely, see the note in Cell 3
#          on running keyword-only recall.
#
# SCOPE (be honest with stakeholders):
#   ANSWERABLE : cohort, % of deviations, root causes, CAPAs, programs, impact text
#   NOT here   : audits/inspections, CTMS/RBQM/QTL linkage, clean FY trends,
#                geography/phase/CRO breakdowns, CAPA effectiveness over time
#   Those need data sources not present in tw_deviation_data_formatted_rdq.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Config
# ─────────────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F, types as T

CATALOG, ALYT   = "us_gmsgq_dev", "gms_us_alyt"
EMBED_INPUT     = f"{CATALOG}.{ALYT}.deviation_embed_input"     # clean text, per pr_id
EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"      # vectors, per pr_id
EMB_INDEX_FINE  = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
VS_ENDPOINT     = "deviation-retrieval-vs"
MAX_LEN         = 640                                            # MUST match indexing

# Output cohort table (write to analytics layer — you have write there)
UNBLINDING_COHORT = f"{CATALOG}.{ALYT}.unblinding_cohort"

print("Config loaded. Source:", EMBED_INPUT)


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Load BGE-M3 (only needed for the semantic half of retrieval)
# ─────────────────────────────────────────────────────────────────────────
from FlagEmbedding import BGEM3FlagModel
print("Loading BAAI/bge-m3 …")
bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("Model loaded.")

def embed_query(text: str) -> list:
    out = bge.encode([text], batch_size=1, max_length=MAX_LEN,
                     return_dense=True, return_sparse=False, return_colbert_vecs=False)
    return out["dense_vecs"][0].tolist()

In [0]:
%pip install databricks-ai-search


In [0]:
%pip install databricks

In [0]:



# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — HYBRID RETRIEVAL: build a high-recall unblinding cohort
#
# Similarity search alone has a threshold problem (where do you cut the score?).
# So we combine TWO signals and union them:
#   (A) SEMANTIC  — several unblinding query phrasings vs. the FINE index
#   (B) KEYWORD   — literal unblinding terms in the deviation text (high precision)
# A pr_id in EITHER set enters the cohort. This maximizes recall while keeping
# keyword hits as a precision anchor.
# ─────────────────────────────────────────────────────────────────────────
import sys, os, databricks
for _sp in sys.path:
    _dbp = os.path.join(_sp, "databricks")
    if os.path.isdir(os.path.join(_dbp, "ai_search")) and _dbp not in databricks.__path__:
        databricks.__path__.append(_dbp)
        break
from databricks.ai_search.client import AISearchClient
client = AISearchClient()

# --- (A) Semantic: multiple phrasings to cover different ways unblinding shows up
UNBLINDING_QUERIES = [
    "inadvertent unblinding of treatment assignment",
    "site accidentally disclosed which arm the subject was on",
    "blinding was broken / code break performed without authorization",
    "unblinded data shared with blinded personnel",
    "randomization or IP kit revealed treatment allocation",
    "pharmacist or monitor became unblinded",
]
SEM_PER_QUERY   = 50      # pull top-N per phrasing (tune for recall)
SEM_SCORE_FLOOR = 0.45    # drop weak semantic matches (tune after inspecting)

sem_ids = set()
idx = client.get_index(endpoint_name=VS_ENDPOINT, index_name=EMB_INDEX_FINE)
for q in UNBLINDING_QUERIES:
    res = idx.similarity_search(
        query_vector=embed_query(q),
        columns=["pr_id"],
        num_results=SEM_PER_QUERY,
    )
    rows = res.get("result", {}).get("data_array", []) if isinstance(res, dict) else []
    for row in rows:
        pr_id, score = row[0], row[-1]
        if score is not None and score >= SEM_SCORE_FLOOR:
            sem_ids.add(str(pr_id))
print(f"(A) semantic cohort: {len(sem_ids):,} unique pr_id")

# --- (B) Keyword: literal unblinding language (high precision anchor)
#     NOTE: if you want to skip GPU entirely, run ONLY this block — it needs no bge.
KW = r"(?i)\b(unblind|un-blind|unblinding|unblinded|" \
     r"treatment assignment (was )?(disclosed|revealed|broken)|" \
     r"code break|broke the blind|blind was broken|" \
     r"randomi[sz]ation.*(reveal|disclos)|allocation.*(reveal|disclos))\b"

src = spark.table(EMBED_INPUT)
kw_df = src.filter(F.col("combined_text").rlike(KW)).select("pr_id")
kw_ids = set(str(r["pr_id"]) for r in kw_df.collect())
print(f"(B) keyword cohort : {len(kw_ids):,} unique pr_id")

# --- Union → cohort, tag provenance so you can inspect precision later
cohort_ids = sem_ids | kw_ids
print(f"UNION cohort       : {len(cohort_ids):,} unique pr_id")
print(f"  both signals     : {len(sem_ids & kw_ids):,}")
print(f"  semantic only    : {len(sem_ids - kw_ids):,}")
print(f"  keyword only     : {len(kw_ids - sem_ids):,}")

cohort_meta = spark.createDataFrame(
    [(pid, pid in sem_ids, pid in kw_ids) for pid in cohort_ids],
    schema=T.StructType([
        T.StructField("pr_id", T.StringType()),
        T.StructField("hit_semantic", T.BooleanType()),
        T.StructField("hit_keyword",  T.BooleanType()),
    ]),
)


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Join cohort back to full deviation text + write cohort table
# ─────────────────────────────────────────────────────────────────────────
# Pull the rich columns from the ORIGINAL source (not just embed_input) so we
# have root cause / impact / action fields for profiling.
SRC_TABLE = f"{CATALOG}.{ALYT.replace('alyt','mart') if False else 'gms_us_mart'}.tw_deviation_data_formatted_rdq"
# ^ adjust if your mart schema name differs; falls back to embed_input columns below.

try:
    raw = spark.table(SRC_TABLE)
    have_raw = True
except Exception:
    have_raw = False
    print("NOTE: could not read source mart table; profiling from embed_input text only.")

cohort = cohort_meta.join(spark.table(EMBED_INPUT), on="pr_id", how="left")

if have_raw:
    # event-grain the raw table the same way the build did (first non-null per event)
    keep = ["Event_Number", "Program_Number", "Study_Protocol",
            "Root_Cause_Category", "Root_Cause_SubCategory",
            "Impact_Assessment", "Quality_Final_Assessment", "Action_Text"]
    keep = [c for c in keep if c in raw.columns]
    raw_event = (raw.groupBy(F.col("Event_Number").cast("string").alias("pr_id"))
                    .agg(*[F.first(F.col(c), ignorenulls=True).alias(c)
                           for c in keep if c != "Event_Number"]))
    cohort = cohort.join(raw_event, on="pr_id", how="left")

(cohort.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(UNBLINDING_COHORT))
print(f"✓ wrote {UNBLINDING_COHORT}: {spark.table(UNBLINDING_COHORT).count():,} rows")
display(spark.table(UNBLINDING_COHORT).limit(20))


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Q: "% of unblinding issues"  (cohort as share of all deviations)
# ─────────────────────────────────────────────────────────────────────────
total_dev = spark.table(EMBED_INPUT).count()
cohort_n  = spark.table(UNBLINDING_COHORT).count()
print(f"Total deviations        : {total_dev:,}")
print(f"Unblinding cohort       : {cohort_n:,}")
print(f"Unblinding as % of all  : {100.0*cohort_n/total_dev:.1f}%")
# Precision caveat: keyword-only hits are high-confidence; semantic-only hits
# should be spot-checked before quoting the % externally.


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Q: impacted programs / protocols + recurrence
# ─────────────────────────────────────────────────────────────────────────
c = spark.table(UNBLINDING_COHORT)
if "Program_Number" in c.columns:
    print("Top impacted programs (recurrence = systemic signal):")
    display(c.groupBy("Program_Number").count().orderBy(F.col("count").desc()).limit(25))
if "Study_Protocol" in c.columns:
    print("Top impacted protocols:")
    display(
        c.withColumn("Study_Protocol",
            F.when(F.trim(F.col("Study_Protocol")) == "", F.lit("(no protocol)"))
             .otherwise(F.col("Study_Protocol")))
        .groupBy("Study_Protocol").count().orderBy(F.col("count").desc()).limit(25)
    )


# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Q: predominant/recurring ROOT CAUSES
# ─────────────────────────────────────────────────────────────────────────
if "Root_Cause_Category" in c.columns:
    print("Root cause categories among unblinding events:")
    display(c.groupBy("Root_Cause_Category").count().orderBy(F.col("count").desc()))
if "Root_Cause_SubCategory" in c.columns:
    print("Root cause sub-categories:")
    display(c.groupBy("Root_Cause_SubCategory").count().orderBy(F.col("count").desc()).limit(30))


# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Q: remediations (CAPAs) — surface the Action_Text for review
# ─────────────────────────────────────────────────────────────────────────
if "Action_Text" in c.columns:
    print("Sample CAPAs / remediations (manual review — free text):")
    display(c.select("pr_id", "Program_Number", "Action_Text")
             .filter(F.length(F.trim("Action_Text")) > 0)
             .limit(40))
# Recurrence proxy: programs with >1 unblinding event = candidates for
# "are CAPAs actually preventing recurrence?" (full answer needs CAPA status data).
if "Program_Number" in c.columns:
    print("Programs with repeat unblinding events (recurrence candidates):")
    display(c.groupBy("Program_Number").count()
             .filter(F.col("count") > 1).orderBy(F.col("count").desc()))


# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Q: impact on subject safety & data integrity (PARTIAL — free text)
# ─────────────────────────────────────────────────────────────────────────
c_cols = set(c.columns)  # hoist schema read outside loop (Spark Connect)
for col in ("Impact_Assessment", "Quality_Final_Assessment"):
    if col in c_cols:
        print(f"--- {col} (sample; free text, inconsistent) ---")
        display(c.select("pr_id", "Program_Number", col)
                 .filter(F.length(F.trim(col)) > 0).limit(25))
# Light signal: flag events whose impact text mentions safety / data integrity
if "Impact_Assessment" in c.columns:
    flagged = c.withColumn(
        "mentions_safety",   F.col("Impact_Assessment").rlike(r"(?i)safety|subject|patient|harm")
    ).withColumn(
        "mentions_integrity", F.col("Impact_Assessment").rlike(r"(?i)data integrity|reliab|validity|GCP")
    )
    display(flagged.select(
        F.sum(F.col("mentions_safety").cast("int")).alias("impact_mentions_safety"),
        F.sum(F.col("mentions_integrity").cast("int")).alias("impact_mentions_integrity"),
        F.count("*").alias("cohort_total"),
    ))


In [0]:
# ─────────────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F, types as T
import matplotlib.pyplot as plt

c_pd = spark.table(UNBLINDING_COHORT).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Inadvertent Unblinding — Cohort Overview", fontsize=14, fontweight="bold")

# — Panel 1: cohort provenance
sem_only = ((c_pd["hit_semantic"]) & ~(c_pd["hit_keyword"])).sum()
kw_only  = (~(c_pd["hit_semantic"]) & (c_pd["hit_keyword"])).sum()
both     = ((c_pd["hit_semantic"]) & (c_pd["hit_keyword"])).sum()
wedges, texts, autotexts = axes[0].pie(
    [sem_only, kw_only, both],
    labels=["Semantic only", "Keyword only", "Both signals"],
    autopct="%1.0f%%",
    colors=["#4C72B0", "#DD8452", "#55A868"],
    startangle=90,
)
for at in autotexts: at.set_fontsize(9)
axes[0].set_title(f"Cohort Provenance  (n={len(c_pd):,})")

# — Panel 2: root cause categories
if "Root_Cause_Category" in c_pd.columns:
    rc = c_pd["Root_Cause_Category"].fillna("Unknown").value_counts().head(10)
    axes[1].barh([str(t)[:35] for t in rc.index[::-1]], rc.values[::-1], color="#4C72B0")
    axes[1].set_title("Top Root Cause Categories")
    axes[1].set_xlabel("Events")
    axes[1].tick_params(axis="y", labelsize=8)
else:
    axes[1].set_visible(False)

# — Panel 3: top impacted programs
if "Program_Number" in c_pd.columns:
    prog = c_pd["Program_Number"].fillna("Unknown").value_counts().head(10)
    axes[2].barh([str(p)[:30] for p in prog.index[::-1]], prog.values[::-1], color="#55A868")
    axes[2].set_title("Top Impacted Programs")
    axes[2].set_xlabel("Events")
    axes[2].tick_params(axis="y", labelsize=8)
else:
    axes[2].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

c_pd = spark.table(UNBLINDING_COHORT).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Root Cause Detail", fontsize=13, fontweight="bold")

if "Root_Cause_Category" in c_pd.columns and "Root_Cause_SubCategory" in c_pd.columns:
    # left: stacked bar — top 8 categories broken down by sub-category
    top_cats = c_pd["Root_Cause_Category"].fillna("Unknown").value_counts().head(8).index.tolist()
    sub_counts = (
        c_pd[c_pd["Root_Cause_Category"].isin(top_cats)]
        .assign(sub=lambda d: d["Root_Cause_SubCategory"].fillna("Unknown"))
        .groupby(["Root_Cause_Category", "sub"]).size()
        .unstack(fill_value=0)
    )
    sub_counts.loc[[c for c in top_cats if c in sub_counts.index]].plot(
        kind="barh", stacked=True, ax=axes[0], colormap="tab20", legend=True
    )
    axes[0].set_title("Category × Sub-category")
    axes[0].set_xlabel("Events")
    axes[0].tick_params(axis="y", labelsize=8)
    axes[0].legend(fontsize=7, loc="upper right")

    # right: top sub-categories standalone
    sc = c_pd["Root_Cause_SubCategory"].fillna("Unknown").value_counts().head(15)
    axes[1].barh([str(s)[:45] for s in sc.index[::-1]], sc.values[::-1], color="#C44E52")
    axes[1].set_title("Top Sub-categories (all events)")
    axes[1].set_xlabel("Events")
    axes[1].tick_params(axis="y", labelsize=8)
elif "Root_Cause_SubCategory" in c_pd.columns:
    sc = c_pd["Root_Cause_SubCategory"].fillna("Unknown").value_counts().head(15)
    axes[0].barh([str(s)[:45] for s in sc.index[::-1]], sc.values[::-1], color="#C44E52")
    axes[0].set_title("Root Cause Sub-categories")
    axes[0].set_xlabel("Events")
    axes[1].set_visible(False)
else:
    for ax in axes: ax.set_visible(False)
    print("Root cause sub-category column not found in cohort.")

plt.tight_layout()
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

c_pd = spark.table(UNBLINDING_COHORT).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Recurrence Analysis — Programs with Repeat Unblinding Events", fontsize=13, fontweight="bold")

if "Program_Number" in c_pd.columns:
    prog_counts = c_pd["Program_Number"].fillna("Unknown").value_counts()

    # left: how many programs have 1, 2, 3... events
    axes[0].hist(prog_counts.values, bins=range(1, int(prog_counts.max()) + 2),
                 color="#4C72B0", edgecolor="white", align="left", rwidth=0.8)
    axes[0].set_title("Distribution: Events per Program")
    axes[0].set_xlabel("# Unblinding Events")
    axes[0].set_ylabel("# Programs")
    axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # right: programs with >1 event (recurrence candidates)
    recur = prog_counts[prog_counts > 1].head(15)
    if len(recur) > 0:
        bars = axes[1].barh([str(p)[:30] for p in recur.index[::-1]], recur.values[::-1], color="#C44E52")
        axes[1].set_title(f"Programs with ≥2 Events  (n={len(recur)}  programs)")
        axes[1].set_xlabel("# Unblinding Events")
        axes[1].tick_params(axis="y", labelsize=8)
        for bar, v in zip(bars, recur.values[::-1]):
            axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                         str(v), va="center", fontsize=9)
    else:
        axes[1].text(0.5, 0.5, "No programs with >1 unblinding event",
                     ha="center", va="center", transform=axes[1].transAxes, fontsize=11)
        axes[1].set_title("Programs with Repeat Events")
else:
    for ax in axes: ax.set_visible(False)
    print("Program_Number column not found.")

plt.tight_layout()
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

c_pd = spark.table(UNBLINDING_COHORT).toPandas()

if "Impact_Assessment" in c_pd.columns:
    ia = c_pd["Impact_Assessment"].fillna("")
    total = len(c_pd)
    flags = {
        "Safety / patient harm": ia.str.contains(r"(?i)safety|subject|patient|harm", regex=True).sum(),
        "Data integrity / GCP": ia.str.contains(r"(?i)data integrity|reliab|validity|GCP", regex=True).sum(),
        "Regulatory / authority": ia.str.contains(r"(?i)regulator|authority|FDA|EMA|ICH", regex=True).sum(),
        'Stated "no impact"': ia.str.contains(r"(?i)no (direct )?impact|minimal|negligible", regex=True).sum(),
        "Impact field blank": (ia.str.strip() == "").sum(),
    }
    colors = ["#C44E52", "#DD8452", "#4C72B0", "#55A868", "#BBBBBB"]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle("Impact Assessment — Keyword Flags across Unblinding Cohort", fontsize=13, fontweight="bold")

    # left: horizontal bar with counts
    labels = list(flags.keys())
    counts = list(flags.values())
    bars = axes[0].barh(labels[::-1], counts[::-1], color=colors[::-1])
    axes[0].set_title("Event Count")
    axes[0].set_xlabel("Events")
    for bar, cnt in zip(bars, counts[::-1]):
        axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     f"{cnt}  ({100*cnt/total:.0f}%)", va="center", fontsize=9)
    axes[0].set_xlim(0, max(counts) * 1.35)

    # right: donut — share of cohort per flag
    axes[1].pie(
        counts,
        labels=labels,
        autopct=lambda p: f"{p:.0f}%" if p > 3 else "",
        colors=colors,
        startangle=90,
        wedgeprops={"width": 0.5},
    )
    axes[1].set_title(f"Share of Cohort  (n={total:,})\n(flags are not mutually exclusive)")

    plt.tight_layout()
    plt.show()
else:
    print("Impact_Assessment column not found in cohort.")